# Firestore Reader Notebook

This notebook demonstrates how to authenticate with Google Cloud Firestore and read data from an existing instance.

## Prerequisites
- Google Cloud Project with Firestore enabled
- Authentication credentials (Service Account Key or Application Default Credentials)
- google-cloud-firestore package installed

## Authentication Options

Choose one of the authentication methods below:

## Method 2: Application Default Credentials (ADC)

This method uses credentials from your environment. Run these commands in your terminal first:

```bash
# Install Google Cloud SDK if you haven't already
# Then run:
gcloud auth application-default login
gcloud config set project som-rit-phi-mhc-dev
```

If you've already run these commands and still get an error, try the service account method above instead.

In [ ]:
from google.cloud import firestore

# Your Google Cloud Project ID
PROJECT_ID = "som-rit-phi-mhc-prod"


def authenticate_with_adc():
    """Authenticate using Application Default Credentials."""
    return firestore.Client(project=PROJECT_ID)


# Use this method if you have ADC set up
db = authenticate_with_adc()
print("Authenticated successfully with Application Default Credentials!")

## Reading Data from Firestore

Once authenticated, you can read data from your Firestore collections.

In [ ]:
def list_collections():
    """List all collections in your Firestore database."""
    collections = db.collections()
    print("Available collections:")
    for collection in collections:
        print(f"- {collection.id}")
    return [collection.id for collection in collections]


# List all collections
collections = list_collections()

In [ ]:
def list_all_collections_and_subcollections():
    """Recursively list all collections and sub-collections in the database."""

    def get_subcollections_recursive(collection_ref, path=""):
        """Helper function to recursively get all collections and sub-collections."""
        collections_info = []

        # Get documents in this collection to check for sub-collections
        docs = collection_ref.stream()

        for doc in docs:
            # Check if this document has any sub-collections
            subcollections = doc.reference.collections()
            for subcollection in subcollections:
                sub_path = f"{path}/{collection_ref.id}/{doc.id}/{subcollection.id}"
                collections_info.append(sub_path)
                # Recursively get deeper sub-collections
                deeper_collections = get_subcollections_recursive(
                    subcollection, sub_path
                )
                collections_info.extend(deeper_collections)

        return collections_info

    all_collections = []

    # Get top-level collections
    top_level = db.collections()
    for collection in top_level:
        collection_path = collection.id
        all_collections.append(collection_path)

        # Get sub-collections for this collection
        sub_collections = get_subcollections_recursive(collection, collection_path)
        all_collections.extend(sub_collections)

    # Display results
    print("All collections and sub-collections:")
    print("=" * 50)
    for collection_path in sorted(all_collections):
        indent_level = collection_path.count("/")
        indent = "  " * indent_level
        print(f"{indent}- {collection_path}")

    print(f"\nTotal collections found: {len(all_collections)}")
    return all_collections


# List all collections and sub-collections
all_collections = list_all_collections_and_subcollections()

In [ ]:
def read_collection(collection_name, limit=10):
    """Read documents from a specific collection."""
    collection_ref = db.collection(collection_name)
    docs = collection_ref.limit(limit).stream()

    print(f"Reading {limit} documents from collection '{collection_name}':")
    print("=" * 50)

    for doc in docs:
        print(f"Document ID: {doc.id}")
        print(f"Data: {doc.to_dict()}")
        print("-" * 30)


read_collection("users")

In [ ]:
def list_collection_contents(collection_name):
    """List all documents and direct sub-collections in a collection."""
    collection_ref = db.collection(collection_name)

    documents = []
    subcollections = set()  # Use set to avoid duplicates

    # Get all documents and their sub-collections
    docs = collection_ref.stream()
    for doc in docs:
        documents.append(doc.id)

        # Check for direct sub-collections
        doc_subcollections = doc.reference.collections()
        for subcollection in doc_subcollections:
            subcollections.add(subcollection.id)

    print(f"Contents of collection '{collection_name}':")
    print("=" * 50)

    print(f"Documents ({len(documents)}):")
    for doc_id in sorted(documents):
        print(f"  📄 {doc_id}")

    print(f"\nDirect sub-collections ({len(subcollections)}):")
    for subcoll_name in sorted(subcollections):
        print(f"  📁 {subcoll_name}/")

    return {"documents": documents, "subcollections": list(subcollections)}


list_collection_contents("users/WhTFb0Ok3jeqate1lVSSS8avnPv1")

In [ ]:
def list_document_subcollections(document_path):
    """List all sub-collections directly under a specific document."""

    # Split the path into collection and document ID
    path_parts = document_path.split("/")
    if len(path_parts) % 2 == 1:  # Odd length means it ends with collection
        raise ValueError("Path must point to a document, not a collection")

    # Build the document reference
    doc_ref = db.document(document_path)

    subcollections = []
    try:
        for collection in doc_ref.collections():
            subcollections.append(collection.id)
    except Exception as e:
        print(f"No document found at path: {document_path}")
        return []

    print(f"Sub-collections under document '{document_path}':")
    print("=" * 50)
    if subcollections:
        for subcoll in sorted(subcollections):
            print(f"  📁 {subcoll}/")
    else:
        print("  (none)")

    return subcollections


# Check if a specific user has sub-collections
subcollections = list_document_subcollections("users/WhTFb0Ok3jeqate1lVSSS8avnPv1")

In [ ]:
subcoll_path = "users/WhTFb0Ok3jeqate1lVSSS8avnPv1/HealthObservations_com.apple.SensorKit.motion.accelerometer"
subcoll_ref = db.collection(subcoll_path)

for doc in subcoll_ref.stream():
    print(doc.id, doc.to_dict())

In [ ]:
from google.cloud import firestore

# Your Firestore client setup
db = firestore.Client(project="som-rit-phi-mhc-prod")


# Function to get sensor data for an observation
def get_sensor_data(observation_id):
    # Construct the DocumentReference ID
    ref_id = f"{observation_id}_Ref"

    # Get the DocumentReference
    doc_ref = db.collection(
        "HealthObservations_com.apple.SensorKit.motion.accelerometer"
    ).document(ref_id)
    doc = doc_ref.get()

    if doc.exists:
        data = doc.to_dict()
        attachment = data["content"][0]["attachment"]
        file_url = attachment["url"]

        # Now you'd download and decompress the file
        # (You'll need to implement the actual download/decompression logic)
        print(f"Data file URL: {file_url}")
        print(f"File size: {attachment['size']} bytes")
        print(f"Content type: {attachment['contentType']}")

        return file_url
    else:
        print(f"DocumentReference {ref_id} not found")
        return None


# Use it with your observation
observation_id = "2026-01-08T05:57:47Z_2026-01-08T06:14:29Z"
file_url = get_sensor_data(observation_id)

In [ ]:
def read_document(collection_name, document_id):
    """Read a specific document from a collection."""
    doc_ref = db.collection(collection_name).document(document_id)
    doc = doc_ref.get()

    if doc.exists:
        print(f"Document '{document_id}' in collection '{collection_name}':")
        print(doc.to_dict())
        return doc.to_dict()
    else:
        print(f"Document '{document_id}' not found in collection '{collection_name}'")
        return None


read_document("users", "WhTFb0Ok3jeqate1lVSSS8avnPv1")

In [ ]:
def query_collection(collection_name, field, operator, value, limit=10):
    """Query documents in a collection with filters."""
    collection_ref = db.collection(collection_name)
    query = collection_ref.where(field, operator, value).limit(limit)
    docs = query.stream()

    print(f"Query results for {field} {operator} {value} in '{collection_name}':")
    print("=" * 50)

    results = []
    for doc in docs:
        data = doc.to_dict()
        results.append({"id": doc.id, "data": data})
        print(f"Document ID: {doc.id}")
        print(f"Data: {data}")
        print("-" * 30)

    return results


# Example query: find documents where 'status' equals 'active'
# results = query_collection('your-collection-name', 'status', '==', 'active')

In [ ]:
def get_collection_fields(collection_name):
    """Get all unique fields present in a collection."""
    collection_ref = db.collection(collection_name)
    all_fields = set()

    docs = collection_ref.stream()
    for doc in docs:
        data = doc.to_dict()
        all_fields.update(data.keys())

    fields_list = sorted(list(all_fields))
    print(f"Fields in collection '{collection_name}':")
    print("=" * 50)
    for field in fields_list:
        print(f"- {field}")

    print(f"\nTotal unique fields: {len(fields_list)}")
    return fields_list


# Get all fields in a collection
ields = get_collection_fields("users")

In [ ]:
def get_collection_count(collection_name):
    """Get the approximate count of documents in a collection."""
    collection_ref = db.collection(collection_name)
    count = 0
    for _ in collection_ref.stream():
        count += 1

    print(f"Collection '{collection_name}' has approximately {count} documents")
    return count


# Get count of documents in a collection
# count = get_collection_count('your-collection-name')

## Downloading SensorKit Data

The SensorKit data is stored as Zstandard-compressed CSV files in Google Cloud Storage. 
To access this data, we need to:
1. Connect to Google Cloud Storage
2. Download the compressed file
3. Decompress it using `zstandard`
4. Load it into pandas for analysis

In [ ]:
import zstandard
import io
import pandas as pd
from google.cloud import storage

print("Libraries imported successfully")

In [ ]:
def list_buckets(project_id=PROJECT_ID):
    """List all buckets in the project to help identify the correct one."""
    storage_client = storage.Client(project=project_id)
    buckets = list(storage_client.list_buckets())
    print("Available Buckets:")
    for bucket in buckets:
        print(f"- {bucket.name}")
    return buckets


def download_sensorkit_data(gcs_path, project_id=PROJECT_ID):
    """
    Downloads and decompresses a SensorKit data file from GCS.

    Args:
        gcs_path (str): The full GCS path (e.g., gs://bucket-name/path/to/file.csv.zstd)
        project_id (str): The Google Cloud Project ID

    Returns:
        pd.DataFrame: The sensor data as a DataFrame
    """
    # Handle gs:// prefix
    if gcs_path.startswith("gs://"):
        path_parts = gcs_path[5:].split("/", 1)
    else:
        path_parts = gcs_path.split("/", 1)

    if len(path_parts) != 2:
        print("Invalid path format. Expected 'gs://bucket-name/path/to/file'")
        return None

    bucket_name, blob_name = path_parts

    print(f"Connecting to bucket: {bucket_name}")
    storage_client = storage.Client(project=project_id)
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_name)

    if not blob.exists():
        print(f"Error: Blob '{blob_name}' does not exist in bucket '{bucket_name}'")
        return None

    print(f"Downloading {blob_name}...")
    compressed_data = blob.download_as_bytes()

    print("Decompressing data...")
    dctx = zstandard.ZstdDecompressor()
    decompressed_data = dctx.decompress(compressed_data)

    print("Loading into DataFrame...")
    # SensorKit CSVs usually have a header, but let's verify format
    try:
        df = pd.read_csv(io.BytesIO(decompressed_data))
        print(f"Successfully loaded {len(df)} rows")
        return df
    except Exception as e:
        print(f"Error loading CSV: {e}")
        return decompressed_data


# Example usage (commented out):
# list_buckets()
# df = download_sensorkit_data("gs://your-bucket-name/SensorKit/...")
# df.head()

In [ ]:
# Specific example: Download data for the observation mentioned
user_id = "WhTFb0Ok3jeqate1lVSSS8avnPv1"
collection_name = "HealthObservations_com.apple.SensorKit.motion.accelerometer"
doc_ref_id = "2026-01-08T05:57:47Z_2026-01-08T06:14:29Z_Ref"

# Construct the path to the DocumentReference
doc_path = f"users/{user_id}/{collection_name}/{doc_ref_id}"
print(f"Looking for document at: {doc_path}")

doc = db.document(doc_path).get()

if doc.exists:
    data = doc.to_dict()
    # Extract the relative URL from the FHIR DocumentReference structure
    try:
        relative_url = data["content"][0]["attachment"]["url"]
        print(f"Found relative URL: {relative_url}")

        # Construct the full path for the SensorKit file
        target_bucket_name = "som-rit-phi-mhc-prod.firebasestorage.app"
        full_file_path = f"users/{user_id}/{relative_url}"

        print(f"\nTarget bucket: {target_bucket_name}")
        print(f"Full file path: {full_file_path}")

        # Check if SensorKit directory exists for this user
        storage_client = storage.Client(project=PROJECT_ID)
        target_bucket = storage_client.bucket(target_bucket_name)

        # Check if the user's SensorKit directory exists
        sensorkit_prefix = f"users/{user_id}/SensorKit/"
        blobs = list(target_bucket.list_blobs(prefix=sensorkit_prefix, max_results=1))

        if not blobs:
            print(f"❌ SensorKit directory does not exist for user {user_id}")
            print(f"Expected path: gs://{target_bucket_name}/{sensorkit_prefix}")
        else:
            # Check if the specific file exists
            blob = target_bucket.blob(full_file_path)
            if blob.exists():
                full_gcs_path = f"gs://{target_bucket_name}/{full_file_path}"
                print(f"✅ Found file in bucket: {target_bucket_name}")
                print(f"Full GCS Path: {full_gcs_path}")

                # Download and display the data
                print("\nDownloading and processing...")
                df = download_sensorkit_data(full_gcs_path, project_id=PROJECT_ID)

                if df is not None:
                    print("\nData Preview:")
                    display(df.head())
                    print(f"\nShape: {df.shape}")
                else:
                    print("❌ Failed to download and process the data")
            else:
                print(
                    f"❌ File not found at: gs://{target_bucket_name}/{full_file_path}"
                )
                print("Available SensorKit files for this user:")

                # List available files in the user's SensorKit directory
                sensorkit_blobs = list(
                    target_bucket.list_blobs(prefix=sensorkit_prefix)
                )
                if sensorkit_blobs:
                    for sb in sensorkit_blobs[:10]:  # Show first 10 files
                        print(f"  - {sb.name}")
                    if len(sensorkit_blobs) > 10:
                        print(f"  ... and {len(sensorkit_blobs) - 10} more files")
                else:
                    print("  (No files found in SensorKit directory)")

    except KeyError as e:
        print(f"Error parsing document structure: Missing key {e}")
        print("Document data:", data)
else:
    print(f"❌ Document not found at {doc_path}")

In [ ]:
import plotly.graph_objs as go
import pandas as pd

# Check if the required columns exist
required_cols = {"timestamp", "x", "y", "z"}
if required_cols.issubset(df.columns):
    # Convert timestamps to datetime (assume seconds since epoch)
    timestamps_converted = pd.to_datetime(df["timestamp"], unit="s")
    fig = go.Figure()
    fig.add_trace(
        go.Scatter(x=timestamps_converted, y=df["x"], mode="lines", name="x axis")
    )
    fig.add_trace(
        go.Scatter(x=timestamps_converted, y=df["y"], mode="lines", name="y axis")
    )
    fig.add_trace(
        go.Scatter(x=timestamps_converted, y=df["z"], mode="lines", name="z axis")
    )
    fig.update_layout(
        title="SensorKit Axes vs. Time",
        xaxis_title="Time",
        yaxis_title="Value",
        legend_title="Axis",
        width=1200,
        height=400,
    )
    fig.show()
else:
    print("Required columns for plotting not found in DataFrame.")

In [ ]:
# Calculate and print the sampling frequency of the data
if "timestamp" in df.columns and len(df["timestamp"]) > 1:
    # Sort timestamps just in case
    sorted_timestamps = df["timestamp"].sort_values().values
    # Compute time diff in seconds
    time_diffs = pd.Series(sorted_timestamps).diff().dropna()
    if not time_diffs.empty and (time_diffs > 0).all():
        avg_period = time_diffs.mean()
        sampling_frequency = 1 / avg_period if avg_period > 0 else float("nan")
        print(f"Sampling frequency: {sampling_frequency:.2f} Hz")
    else:
        print("Timestamps are not valid or not increasing.")
else:
    print(
        "Cannot calculate sampling frequency: 'timestamp' column missing or not enough data points."
    )

In [ ]:
df.device.iloc[0]